# Financial RAG Agent - Benchmarking & Evaluation Notebook

This notebook provides an automated evaluation framework to test your Financial RAG Agent against ground-truth benchmark datasets.

### Evaluated Metrics:
1. **Correctness**: Accurate numeric extraction, calculations, and answers.
2. **Citation Accuracy**: Verifies if the agent cited the correct source page.
3. **Hallucination Defense**: Verifies that unanswerable questions trigger a proper "not found" refusal.

In [ ]:
import json
import re
from typing import Dict, List, Tuple

## Step 1: Connect Your RAG Agent
Replace the placeholder function below with your actual RAG Agent code.

In [ ]:
def call_rag_agent(question: str) -> Tuple[str, List[int]]:
    """
    TODO FOR MY FRIEND: Replace this dummy function with your real RAG Agent!

    Args:
        question (str): The prompt/question to ask the RAG agent.

    Returns:
        Tuple[str, List[int]]: 
            - str: The text response generated by your agent.
            - List[int]: A list of page numbers retrieved/cited by your agent (e.g., [1, 2]).
    """
    # --- MOCK PLACEHOLDER OUTPUT (Replace with actual agent import/call) ---
    # Example integration:
    # from my_financial_rag import rag_pipeline
    # response = rag_pipeline.query(question)
    # return response.answer_text, response.cited_pages
    
    mock_response = "Information not found in the document."
    mock_cited_pages = []
    return mock_response, mock_cited_pages

## Step 2: Evaluator Class

In [ ]:
class FinancialRAGEvaluator:
    """Evaluates RAG agent outputs against ground-truth benchmarks."""

    UNANSWERABLE_REFUSAL_TERMS = [
        "not found",
        "information not found",
        "not provided",
        "does not state",
        "absent",
        "cannot be found",
    ]

    def __init__(self, benchmark_filepath: str):
        with open(benchmark_filepath, "r", encoding="utf-8") as f:
            self.dataset = json.load(f)

    def evaluate_item(self, item: Dict, agent_response: str, retrieved_pages: List[int]) -> Dict:
        q_type = item["type"]
        expected = item["expected_answer"]
        target_page = item["source_page"]

        metrics = {
            "id": item["id"],
            "type": q_type,
            "question": item["question"],
            "correctness": False,
            "citation_accuracy": False,
            "hallucination_detected": False,
        }

        # 1. Unanswerable Guardrail Check
        if q_type == "unanswerable":
            is_refused = any(term in agent_response.lower() for term in self.UNANSWERABLE_REFUSAL_TERMS)
            metrics["correctness"] = is_refused
            metrics["hallucination_detected"] = not is_refused
            metrics["citation_accuracy"] = (len(retrieved_pages) == 0 or target_page == "N/A")
            return metrics

        # 2. Citation Accuracy Check
        if isinstance(target_page, int):
            metrics["citation_accuracy"] = target_page in retrieved_pages

        # 3. Fact and Number Extraction Accuracy Check
        if q_type in ["direct_lookup", "table_lookup", "derived_metric"]:
            expected_nums = re.findall(r"\d+(?:\.\d+)?", expected.replace(",", ""))
            actual_nums = re.findall(r"\d+(?:\.\d+)?", agent_response.replace(",", ""))
            metrics["correctness"] = any(num in actual_nums for num in expected_nums)
        else:
            metrics["correctness"] = (len(set(expected.lower().split()) & set(agent_response.lower().split())) > 3)

        return metrics

    def run_benchmark(self) -> List[Dict]:
        results = []
        print(f"\n--- Starting Benchmark: {len(self.dataset)} Items ---")
        for item in self.dataset:
            response_text, cited_pages = call_rag_agent(item["question"])
            eval_result = self.evaluate_item(item, response_text, cited_pages)
            results.append(eval_result)
            status = "PASS" if eval_result["correctness"] else "FAIL"
            print(f"[{status}] ID: {item['id']} | Type: {item['type']}")
        return results

## Step 3: Run the Benchmark

In [ ]:
# Load benchmark JSON file and execute evaluation
DATASET_PATH = "dataset_synthetic.json"

evaluator = FinancialRAGEvaluator(DATASET_PATH)
benchmark_results = evaluator.run_benchmark()

# Summary Metrics
total = len(benchmark_results)
correct = sum(1 for r in benchmark_results if r["correctness"])
accurate_citations = sum(1 for r in benchmark_results if r["citation_accuracy"])
hallucinations = sum(1 for r in benchmark_results if r["hallucination_detected"])

print("\n" + "=" * 50)
print(" BENCHMARK PERFORMANCE SUMMARY")
print("=" * 50)
print(f"Total Evaluated Questions : {total}")
print(f"Overall Accuracy          : {(correct / total) * 100:.2f}%")
print(f"Citation Precision        : {(accurate_citations / total) * 100:.2f}%")
print(f"Hallucination Failures    : {hallucinations}")
print("=" * 50 + "\n")